# Project 3 — Context-aware offer recommender

Project 2 asked *what products fit this user?* Project 3 asks *which offer fits this user **right now**?* Every step calls the same `src/` functions as `python -m src.train` and the API.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.data import load_data, load_users

offers, raw_interactions = load_data()
users = load_users()
print("offers:", offers.shape, "interactions:", raw_interactions.shape, "users:", users.shape)
print(offers.columns.tolist())
print(raw_interactions.columns.tolist())

## Inspect the schema
There is no `accepted` column: the response is encoded in `event_type` (impression / click / redeem).

In [ ]:
print(offers.dtypes, "\n")
print(raw_interactions.dtypes, "\n")
print(raw_interactions.isna().sum(), "\n")
print(raw_interactions["event_type"].value_counts())

## Internal schema and label rule
`to_internal_schema` applies `COLUMN_MAP`, parses timestamps and sets `label = 1` for click/redeem. Impressions alone are not treated as rejections.

In [ ]:
from src.config import COLUMN_MAP, POSITIVE_EVENT_TYPES
from src.data import prepare_offers, to_internal_schema

offers = prepare_offers(offers)
interactions = to_internal_schema(raw_interactions)
print(COLUMN_MAP, POSITIVE_EVENT_TYPES)
interactions[["user_id", "offer_id", "timestamp", "event_type", "label"]].head()

## Recommendation-time context
Only things known when the request arrives: device, region, hour, day of week. Hour and day are encoded cyclically.

In [ ]:
from src.features import add_time_features

with_time = add_time_features(interactions)
with_time[["timestamp", "hour", "day_of_week", "is_weekend", "daypart", "hour_sin", "hour_cos"]].head()

## What actually drives engagement?
Check before modeling. Engagement *given* an event barely moves with context, but which offers users interact with is strongly tied to their preferred category.

In [ ]:
events = with_time.merge(offers[["offer_id", "category"]], on="offer_id").merge(users[["user_id", "preferred_category"]], on="user_id")
print(events.groupby("daypart")["label"].mean().round(3), "\n")
print(events.groupby("device_type")["label"].mean().round(3), "\n")
print("share of events in the user's preferred category:", round((events.category == events.preferred_category).mean(), 3))

## Training examples
Positives are click/redeem events in their real context. For each one, 4 other offers are sampled for the same user and context as negatives.

In [ ]:
from src.config import NEGATIVES_PER_POSITIVE, RANDOM_SEED
from src.features import build_training_examples

training_requests = build_training_examples(interactions, offers, NEGATIVES_PER_POSITIVE, RANDOM_SEED)
training_requests["label"].value_counts()

## Feature engineering (point-in-time)
User history, offer popularity and category affinity use only events **before** each row's request time. This is the leakage guard.

In [ ]:
from src.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES, build_feature_frame

training_frame = build_feature_frame(training_requests, offers, users, interactions)
print(NUMERIC_FEATURES)
print(CATEGORICAL_FEATURES)
training_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES + ["label"]].head()

## Train the ranker
Logistic regression: interpretable and fast.

In [ ]:
from src.model import build_model, coefficient_table

ranker = build_model(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
ranker.fit(training_frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES], training_frame["label"])
coefficient_table(ranker).head(12)

## Candidate generation → ranking → Top-K

In [ ]:
from src.candidates import generate_candidates
from src.recommender import ContextualRecommender

candidates, funnel = generate_candidates(offers, region="Lagos", device_type="mobile", today=pd.Timestamp("2026-09-30").date())
print(funnel)

recommender = ContextualRecommender(ranker, NUMERIC_FEATURES, CATEGORICAL_FEATURES, offers, users, interactions)
context = {"device_type": "mobile", "region": "Lagos", "hour": 18, "day_of_week": 2}
result = recommender.recommend("R00001", context, k=10, now=pd.Timestamp("2026-09-30T18:00Z"))
pd.DataFrame(result["recommendations"])[["offer_id", "title", "category", "discount_pct", "score"]]

## Time-aware evaluation vs baselines
Train on the first 80% of time and test on the last 20%. The table compares the contextual model with: context removed, the guide's literal exposure labels, popularity, and a category-diversified list.

In [ ]:
from src.evaluate import evaluate

evaluation = evaluate(interactions, offers, users)
pd.DataFrame(evaluation["models"]).T

In [ ]:
pd.DataFrame(evaluation["by_segment"]["daypart"]).T

## Experiments (guide section 35)

In [ ]:
from src.experiments import main as run_experiments
run_experiments()

## Save the artifact
Same as `python -m src.train`.

In [ ]:
from src.train import train
_, metadata = train()
{key: metadata[key] for key in ["trained_at", "num_interactions", "num_training_rows"]}